# State space generalized linear model (SS-GLM)
> Czanner, G., Eden, U.T., Wirth, S., Yanike, M., Suzuki, W.A., and Brown, E.N. (2008). Analysis of Between-Trial and Within-Trial Neural Spiking Dynamics. Journal of Neurophysiology 99, 2672–2693. https://doi.org/10.1152/jn.00343.2007.


In [2]:
import jax
import jax.numpy as jnp


def log_receptive_field_model(
    position: jnp.ndarray, params: jnp.ndarray
) -> jnp.ndarray:
    log_max_rate, place_field_center, scale = params
    return log_max_rate - (position - place_field_center) ** 2 / (2 * scale**2)


def receptive_field_model(position: jnp.ndarray, params: jnp.ndarray) -> jnp.ndarray:
    return jnp.exp(log_receptive_field_model(position, params))


def stochastic_point_process_filter(
    init_mode_params: jnp.ndarray,
    init_covariance_params: jnp.ndarray,
    x: jnp.ndarray,
    spike_indicator: jnp.ndarray,
    dt: float,
    transition_matrix: jnp.ndarray,
    covariance_matrix: jnp.ndarray,
    log_receptive_field_model: callable,
) -> tuple[jnp.ndarray, jnp.ndarray]:
    """Stochastic State Point Process Filter (SSPPF)

    Parameters
    ----------
    init_mode_params : jnp.ndarray, shape (n_params,)
        Initial mean parameters
    init_covariance_params : jnp.ndarray, shape (n_params, n_params)
        Initial variance parameters
    x : jnp.ndarray, shape (n_time,)
        linear-valued input signal
    spike_indicator : jnp.ndarray, shape (n_time,)
        Spike count
    dt : float
        Time step
    transition_matrix : jnp.ndarray, shape (n_params, n_params)
    covariance_matrix : jnp.ndarray, shape (n_params, n_params)
    log_receptive_field_model : callable
        Function that takes in `x` and parameters and returns the log spike rate

    Returns
    -------
    posterior_mode : jnp.ndarray, shape (n_time, n_params)
    posterior_variance : jnp.ndarray, shape (n_time, n_params, n_params)

    References
    ----------
    ...[1] Eden, U. T., Frank, L. M., Barbieri, R., Solo, V. & Brown, E. N.
      Dynamic Analysis of Neural Encoding by Point Process Adaptive Filtering.
      Neural Computation 16, 971-998 (2004).


    """

    grad_log_receptive_field_model = jax.grad(log_receptive_field_model, argnums=1)
    hess_log_receptive_field_model = jax.hessian(log_receptive_field_model, argnums=1)

    def _step(
        params_prev: tuple[jnp.ndarray, jnp.ndarray],
        args: tuple[jnp.ndarray, jnp.ndarray],
    ) -> tuple[tuple[jnp.ndarray, jnp.ndarray], tuple[jnp.ndarray, jnp.ndarray]]:
        """Point Process Adaptive Filter update step

        F : transition matrix
        Q : covariance matrix
        \theta_{k | k-1} :
        W_{k | k-1}: one_step_variance_params
        \theta_{k | k} : posterior_mode
        W_{k | k} : posterior_variance
        """

        # Unpack previous parameters
        mode_prev, variance_prev = params_prev
        x_t, spike_indicator_t = args

        # One-step prediction
        one_step_mode = transition_matrix @ mode_prev
        one_step_covariance = (
            transition_matrix @ variance_prev @ transition_matrix.T + covariance_matrix
        )

        # Compute the conditional intensity and innovation
        conditional_intensity = (
            jnp.exp(log_receptive_field_model(x_t, one_step_mode)) * dt
        )
        innovation = spike_indicator_t - conditional_intensity

        # Compute the posterior mean and variance
        one_step_grad = grad_log_receptive_field_model(x_t, one_step_mode)[None]
        one_step_hess = hess_log_receptive_field_model(x_t, one_step_mode)

        inverse_posterior_covariance = (
            jnp.linalg.pinv(one_step_covariance)
            + (one_step_grad.T * conditional_intensity @ one_step_grad)
            - innovation * one_step_hess
        )
        posterior_covariance = jnp.linalg.pinv(inverse_posterior_covariance)
        posterior_mode = one_step_mode + posterior_covariance @ (
            one_step_grad.squeeze() * innovation
        )

        return (posterior_mode, posterior_covariance), (
            posterior_mode,
            posterior_covariance,
            one_step_mode,
            one_step_covariance,
        )

    posterior_mode, posterior_covariance, one_step_mode, one_step_covariance = (
        jax.lax.scan(
            _step, (init_mode_params, init_covariance_params), (x, spike_indicator)
        )[1]
    )

    return (
        posterior_mode,
        posterior_covariance,
        one_step_mode,
        one_step_covariance,
    )


def stochastic_point_process_smoother(
    filtered_posterior_mode,
    filtered_posterior_covariance,
    one_step_mode,
    one_step_covariance,
):

    def _step(params_prev, k):
        mode_smoothed_next, covariance_smoothed_next = params_prev
        A_k = filtered_posterior_covariance[k] @ jnp.linalg.pinv(
            one_step_covariance[k + 1]
        )

        mode_smoothed = filtered_posterior_mode[k] + A_k @ (
            mode_smoothed_next - one_step_mode[k + 1]
        )

        covariance_smoothed = (
            filtered_posterior_covariance[k]
            + A_k @ (covariance_smoothed_next - one_step_covariance[k + 1]) @ A_k.T
        )

        return (mode_smoothed, covariance_smoothed), (
            mode_smoothed,
            covariance_smoothed,
            A_k,
        )

    init_params = (filtered_posterior_mode[-1], filtered_posterior_covariance[-1])
    n_time = len(filtered_posterior_mode)
    (_, _), (mode_smoothed, covariance_smoothed, A) = jax.lax.scan(
        _step,
        init_params,
        jnp.arange(n_time - 1),
        reverse=True,
    )

    mode_smoothed = jnp.concatenate([mode_smoothed, filtered_posterior_mode[-1:]])
    covariance_smoothed = jnp.concatenate(
        [covariance_smoothed, filtered_posterior_covariance[-1:]]
    )

    return mode_smoothed, covariance_smoothed, A